# T2-E1 — observer sufficiency

This exploratory study compares CLS history with current CLS and all-token observations on all six frozen U7C models. The original competence failures remain part of the record. No base-model training or reserved confirmation seeds are used. Remaining loss reduction is the primary target; decision flips are descriptive only.

Upload `T2_E1_Observer_Study.zip`. The compact package contains the frozen protocol, six model checkpoints, source, generated data, test predictions, complete results, and method checks. CPU is sufficient. Allow several minutes for reproduction. No external datasets or model downloads are needed.

All steps and both members of a counterfactual family share a split. Test labels never select hyperparameters. The results are exploratory and specific to these exposed, auxiliary-supervised models.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, subprocess, sys, hashlib, json

files.upload()
archive = Path('T2_E1_Observer_Study.zip')
if not archive.is_file():
    raise FileNotFoundError('Upload T2_E1_Observer_Study.zip')
with zipfile.ZipFile(archive) as z:
    if any(Path(n).is_absolute() or '..' in Path(n).parts for n in z.namelist()):
        raise ValueError('Unsafe archive path')
    z.extractall('study_workspace')
root = Path('study_workspace/T2_E1_Observer_Study').resolve()
manifest = json.loads((root / 'PACKAGE_SHA256.json').read_text())
for name, expected in manifest.items():
    if hashlib.sha256((root / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f'Package integrity failure: {name}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)

In [ ]:
# Reproduction writes a new result directory and preserves distributed results.
subprocess.run([sys.executable, str(root / 'verify_methods.py')], check=True)
out = root / 'reproduced_results'
subprocess.run([sys.executable, '-u', str(root / 'run_study.py'), '--out', str(out)], check=True)

In [ ]:
import pandas as pd
result = json.loads((out / 'study_result.json').read_text())
print('Status:', result['status'])
print('Exploratory screen passed:', result['primary_exploratory_screen_passed'])
print('Base-model training updates:', result['training_updates'])
print('Confirmation advancement:', result['confirmation_advancement'])
display(pd.DataFrame(json.loads((out / 'panel_comparisons.json').read_text())))
display(pd.read_csv(out / 'target_diagnostics.csv'))